In [0]:
%pip install azure-storage-blob

In [0]:
%run ./utils

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType
from pyspark.sql.window import Window
import datetime

In [0]:
# Configuration
STORAGE_KEY = configure_spark_azure()
DATE = get_processing_date(fallback_days=4)

silver_path = get_adls_path(CONTAINER_SILVER) + "stock_data/"
gold_path = get_adls_path(CONTAINER_GOLD) + "stock_data/"
gold_summary_path = get_adls_path(CONTAINER_GOLD) + "stock_summary/"

print(f"Silver path: {silver_path}")
print(f"Gold path: {gold_path}")

In [0]:

# # Configuration
# STORAGE_ACCOUNT = "stockpipelinedl"
# STORAGE_KEY = dbutils.secrets.get(scope="stock-pipeline", key="azure-storage-key")
# CONTAINER_SILVER = "silver"
# CONTAINER_GOLD = "gold"

# try:
#     DATE = dbutils.widgets.get("execution_date")
# except:
#     DATE = (datetime.datetime.utcnow() - datetime.timedelta(days=1)).strftime('%Y-%m-%d')

# print(f"Processing date: {DATE}")

# # Connect Spark to Azure Storage
# spark.conf.set(
#     f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
#     STORAGE_KEY
# )

# print("Spark is connected to Azure Storage")

In [0]:
# Read entire Silver Delta table (all dates)
silver_path = f"abfss://{CONTAINER_SILVER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/stock_data/"

silver_df = spark.read.format("delta").load(silver_path)

print(f"Done reading {silver_df.count()} total records from Silver")
print(f"   Dates available: {silver_df.select('trade_date').distinct().count()}")
print("\nSample data:")
silver_df.display(truncate=False)

In [0]:
# Window partitioned by ticker, ordered by date (This means calculations happen per stock, in date order)
ticker_window = Window.partitionBy("ticker").orderBy("trade_date")

# Window for 7-day moving average (rows between 6 preceding and current row = last 7 days including today)
moving_avg_window = Window.partitionBy("ticker") \
    .orderBy("trade_date") \
    .rowsBetween(-6, 0)

# Calculate metrics
gold_df = silver_df \
    .withColumn(
        "prev_close",
        F.lag("close_price", 1).over(ticker_window)
    ) \
    .withColumn(
        "daily_return_pct",
        F.when(
            F.col("prev_close").isNotNull(),
            F.round(
                ((F.col("close_price") - F.col("prev_close")) / F.col("prev_close")) * 100,
                2
            )
        ).otherwise(0)  # Use 0 for first day of stock
    ) \
    .withColumn(
        "moving_avg_7day",
        F.round(F.avg("close_price").over(moving_avg_window), 2)
    ) \
    .withColumn(
        "price_volatility",
        F.round(
            F.col("high_price") - F.col("low_price"),
            2
        )
    ) \
    .withColumn(
        "price_range_pct",
        F.round(
            ((F.col("high_price") - F.col("low_price")) / F.col("low_price")) * 100,
            2
        )
    ) \
    .withColumn("processed_timestamp", F.current_timestamp())

# Select final Gold columns
gold_df = gold_df.select(
    "ticker",
    "trade_date",
    "open_price",
    "high_price",
    "low_price",
    "close_price",
    "volume",
    "daily_return_pct",
    "moving_avg_7day",
    "price_volatility",
    "price_range_pct",
    "processed_timestamp"
)

print(f"Gold metrics calculated!")
print(f"Records: {gold_df.count()}")
print("\nGold data:")
gold_df.display(truncate=False)

In [0]:
# Write to Gold as Delta table
gold_path = f"abfss://{CONTAINER_GOLD}@{STORAGE_ACCOUNT}.dfs.core.windows.net/stock_data/"

gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("trade_date") \
    .save(gold_path)

print(f"Gold Delta table written to ADLS!")
print(f"Path: {gold_path}")
print(f"Records written: {gold_df.count()}")

# Verify by reading back
verify_df = spark.read.format("delta").load(gold_path)
print(f"\nVerification - Records in Gold: {verify_df.count()}")

from IPython.display import display
display(verify_df.toPandas())

In [0]:
# Register Gold Delta table as Temp View for Spark SQL
gold_df.createOrReplaceTempView("gold_stocks")

print("Temp view 'gold_stocks' created")

# Query 1: Best performing stock each day
print("\n Best Performing Stock Each Day:")
best_performer = spark.sql("""
    SELECT 
        trade_date,
        ticker,
        close_price,
        daily_return_pct,
        RANK() OVER (PARTITION BY trade_date ORDER BY daily_return_pct DESC) as rank
    FROM gold_stocks
    WHERE daily_return_pct IS NOT NULL
""")
from IPython.display import display
display(best_performer.toPandas())

# Query 2: Average metrics per ticker
print("\n Average Metrics Per Stock:")
avg_metrics = spark.sql("""
    SELECT
        ticker,
        ROUND(AVG(close_price), 2) AS avg_close_price,
        ROUND(AVG(daily_return_pct), 2) AS avg_daily_return_pct,
        ROUND(AVG(price_volatility), 2) AS avg_volatility,
        ROUND(MAX(close_price), 2) AS max_close_price,
        ROUND(MIN(close_price), 2) AS min_close_price,
        COUNT(*) AS total_trading_days
    FROM gold_stocks
    GROUP BY ticker
    ORDER BY avg_daily_return_pct DESC
""")
display(avg_metrics.toPandas())

# Query 3: Most volatile stocks
print("\n Most Volatile Stocks:")
most_volatile = spark.sql("""
    SELECT
        ticker,
        trade_date,
        price_volatility,
        price_range_pct,
        RANK() OVER (ORDER BY price_volatility DESC) as volatility_rank
    FROM gold_stocks
    ORDER BY price_volatility DESC
""")
display(most_volatile.toPandas())

# SQL Query 4: Stocks trading above 7-day moving average
print("\n Stocks Trading Above 7-Day Moving Average (Bullish Signal):")
bullish = spark.sql("""
    SELECT
        ticker,
        trade_date,
        close_price,
        moving_avg_7day,
        ROUND(close_price - moving_avg_7day, 2) AS above_avg_by,
        CASE 
            WHEN close_price > moving_avg_7day THEN '📈 Bullish'
            WHEN close_price < moving_avg_7day THEN '📉 Bearish'
            ELSE '➡️ Neutral'
        END AS signal
    FROM gold_stocks
    ORDER BY ticker, trade_date
""")
display(bullish.toPandas())

In [0]:
# Write SQL results to Gold as separate Delta table
gold_summary_path = f"abfss://{CONTAINER_GOLD}@{STORAGE_ACCOUNT}.dfs.core.windows.net/stock_summary/"

avg_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_summary_path)

print(f"Gold summary Delta table written!")
print(f"Path: {gold_summary_path}")

# ── Verify ────────────────────────────────────────────────────────────────────
verify_summary = spark.read.format("delta").load(gold_summary_path)
print(f"\nVerification - Records in Gold Summary: {verify_summary.count()}")
display(verify_summary.toPandas())